# **Network Dissection by hand**

A practice for the lesson on Network Dissection. We do not run the authors' code — we assemble the
measure itself out of the three decisions covered in the lesson: a quantile threshold on the
activation, stretching the mask up to the input size, and an IoU accumulated by union over the
whole set rather than averaged over pictures.

## What it costs

**The full Network Dissection** ([NetDissect-Lite](https://github.com/CSAILVision/NetDissect-Lite)):
the Broden dataset takes about **1 GB** of disk; dissecting one network on a GPU takes **about 20
minutes** for ResNet18 and **about two hours** for DenseNet161. The code was written for Python 3.6
and tested on Ubuntu 16.04, the repository has a couple of dozen open issues and no recent updates:
on today's PyTorch versions running it means editing somebody else's code. The method is not buggy —
its stack is old.

**This notebook:** CIFAR-10 is about **170 MB**, the pretrained ResNet18 is **45 MB**, and the whole
computation over 200 images takes **two or three minutes on a CPU** — no GPU needed. What we pay is
that our concepts are not labelled by people but defined by a colour rule. That is a coarsening, and
it is named honestly: the real Broden is labelled pixel by pixel by humans and holds about fifteen
hundred concepts.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("computing on", device)

## Step 1. The network and the set of images

We take ResNet18 trained on ImageNet and look at the last convolutional block, `layer4`: 512
channels and a $7\times7$ map — exactly the resolution about which the lesson says there is nothing
to examine. The images come from CIFAR-10, stretched to 224 pixels as the network expects.

In [ ]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights).eval().to(device)

tf = transforms.Compose([transforms.Resize(224), transforms.ToTensor()])
ds = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=tf)
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(ds, range(200)), batch_size=25)

norm = transforms.Normalize(mean=weights.transforms().mean, std=weights.transforms().std)

acts, imgs = [], []
hook = model.layer4.register_forward_hook(lambda m, i, o: acts.append(o.detach().cpu()))
with torch.no_grad():
    for x, _ in loader:
        imgs.append(x)
        model(norm(x).to(device))
hook.remove()

A = torch.cat(acts)      # activations: (N, K, 7, 7)
X = torch.cat(imgs)      # images:      (N, 3, 224, 224)
print("activations", tuple(A.shape), "| images", tuple(X.shape))

## Step 2. A concept instead of Broden labels

We have no pixel labels, so we define the concept by a rule: a pixel belongs to the concept "green"
if the green channel noticeably exceeds red and blue. This is $L_c(x_i)$ from the formula in the
lesson — the set of pixels "annotated" for concept $c$.

The substitution is not harmless, and that is the whole point of the limitation the lesson talks
about: **we measure not what the network learned, but the overlap of what it learned with our list
of concepts.** A list made of one colour rule is the extreme case of such a list.

In [ ]:
concept = (X[:, 1] > X[:, 0] + 0.06) & (X[:, 1] > X[:, 2] + 0.06)   # L_c(x_i)
print("share of concept pixels over the set:", round(concept.float().mean().item(), 4))

## Step 3. The three decisions hidden in the formula

We assemble the measure exactly as the lesson takes it apart.

1. **The threshold is a quantile, not an absolute number.** $T_k$ is chosen so that
   $P(a_k > T_k) = 0.005$, and it is specific to each channel: activations of different channels
   live in different ranges.
2. **We stretch the mask rather than shrink the labels.** The $7\times7$ map is lifted to
   $224\times224$ by bilinear interpolation. The opposite path would lose everything smaller than a
   $32\times32$ region.
3. **A union over the set, not an average over pictures.** Numerator and denominator accumulate over
   all images, and only then is the ratio taken — otherwise a channel that fired perfectly on three
   pictures out of two hundred would get a high average IoU.

In [ ]:
def iou_for_channel(k, quantile=0.005):
    """IoU of channel k with the concept — by the formula from the lesson."""
    a = A[:, k]
    T = torch.quantile(a.flatten().float(), 1 - quantile)                 # (1) quantile threshold

    mask = F.interpolate(a.unsqueeze(1), size=(224, 224),                 # (2) stretch the mask
                         mode="bilinear", align_corners=False)[:, 0] > T

    inter = (mask & concept).sum().item()                                 # (3) union over the set
    union = (mask | concept).sum().item()
    return inter / union if union else 0.0


scores = np.array([iou_for_channel(k) for k in range(A.shape[1])])
order = scores.argsort()[::-1]

print("channels in total:", len(scores))
print("median IoU over channels:", round(float(np.median(scores)), 5))
for k in order[:5]:
    print(f"  channel {k:>3}: IoU = {scores[k]:.4f}")

Look at the gap: the top channels have an IoU tens of times higher than the median. That is
the claim of the method — the response of individual units really does overlap with a concept
rather than being spread evenly across the network.

Now to the absolute numbers. The detector threshold in the paper is **0.04**. Our best channel most
likely did not reach it. The reason is named in the lesson: bilinear interpolation blurs the edges
of the mask, and the computed IoU comes out lower than the true one — especially for small
objects.

In [ ]:
best = int(order[0])
print(f"best channel: {best}, IoU = {scores[best]:.4f}, detector threshold in the paper is 0.04")
print("would be declared a detector:", scores[best] >= 0.04)

**Task 1.** Raise the share of active pixels from 0.005 to 0.02 (that is, lower the threshold
$T_k$) and recompute. What happens to the IoU of the top channels, and why? The lesson predicts the
answer — check the prediction with a number.

`Your answer here.`

**Task 2.** Replace the concept "green" with "blue" (the blue channel noticeably exceeding red
and green). Do the top channels coincide with those found for green? What would the answer "they
coincide" mean — for the method, and for the claim "one unit — one concept"?

**Task 2.** Replace the concept "green" with "blue" (the blue channel noticeably exceeding red
and green). Do the top channels coincide with those found for green? What would the answer "they
coincide" mean — for the method, and for the claim "one unit — one concept"?

`Your answer here.`

## Step 4. Look with your eyes at what has been computed

It is too early to trust the number before seeing the mask. Below are the images on which the best
channel fired most strongly, and its mask laid over the picture.

## Step 4. Look with your eyes at what has been computed

It is too early to trust the number before seeing the mask. Below are the images on which the best
channel fired most strongly, and its mask laid over the picture.

**Task 3.** Does the mask coincide with the green areas by eye? Find an image where the
channel fired strongly while there is little green in it, and explain what that says about the
caption "channel $k$ is a detector of green".

**Task 3.** Does the mask coincide with the green areas by eye? Find an image where the
channel fired strongly while there is little green in it, and explain what that says about the
caption "channel $k$ is a detector of green".

`Your answer here.`

## What we did and what we did not

We assembled the whole Network Dissection measure and got a meaningful ordering of channels in a
couple of minutes instead of a gigabyte of labels and two hours of computation. What we do not have
is a real vocabulary of concepts. The fifteen hundred human-labelled concepts of Broden are both the
strength of the method and the ceiling it runs into: what is not in the labels, the method will not
see.

The way around that ceiling is covered in the lesson on CLIP-Dissect, and its practice is a separate
notebook, `PRACTICE_clip_dissect`. There the vocabulary is given as text, and there are no masks at
all.

## What we did and what we did not

We assembled the whole Network Dissection measure and got a meaningful ordering of channels in a
couple of minutes instead of a gigabyte of labels and two hours of computation. What we do not have
is a real vocabulary of concepts. The fifteen hundred human-labelled concepts of Broden are both the
strength of the method and the ceiling it runs into: what is not in the labels, the method will not
see.

The way around that ceiling is covered in the lesson on CLIP-Dissect, and its practice is a separate
notebook, `PRACTICE_clip_dissect`. There the vocabulary is given as text, and there are no masks at
all.